# Outdoor Adventure AI Benchmark
### DEV Community × Kaggle Benchmarking Challenge

**Research question:** Which free API-hosted language models most reliably generate useful, varied, safe, and correctly structured outdoor adventures for an app such as Ruta Viva AI?

This notebook is designed to run locally in VS Code or on Kaggle. It uses OpenRouter's API and reads the API key from an environment variable or `.env` file.

## Important research notes
- Model availability and free-tier limits change. The notebook discovers the current catalog and checks model responses before the full run.
- `HTTP 200` alone does not mean a model completed the task correctly. We validate JSON, required fields, mission count, XP consistency, and response content.
- Safety checks are only keyword heuristics, not proof of safety. Review outputs manually before making safety claims.
- The full run can make up to 90 requests (3 models × 30 scenarios). Start with pilot mode.
- Do not publish your API key or commit `.env`. Record errors and actual results honestly.

## 1. Install dependencies

In the VS Code terminal with your project's virtual environment active:

```powershell
python -m pip install requests pandas python-dotenv ipykernel
```

Create a `.env` file beside this notebook:

```dotenv
OPENROUTER_API_KEY=your_real_key_here
```

Ensure `.env` is in `.gitignore`. Never print or publish the key.

On Kaggle, add a secret named `OPENROUTER_API_KEY` under Add-ons → Secrets. Internet access must be enabled for API calls; if Kaggle denies that permission, run this notebook locally and upload the notebook plus selected result files to Kaggle.

In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY was not found. Check that .env is beside the notebook "
        "or configure the secret in Kaggle."
    )

BASE_URL = "https://openrouter.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://www.kaggle.com/",
    "X-Title": "Outdoor Adventure AI Benchmark",
}

# Start small. Set PILOT_MODE = False only after pilot results look reasonable.
PILOT_MODE = True
PILOT_CASES = 3
FULL_CASES = 30
REQUEST_TIMEOUT_SECONDS = 90
PAUSE_BETWEEN_REQUESTS_SECONDS = 2.0
MAX_RETRIES = 1

print("API key loaded:", bool(API_KEY))
print("Pilot mode:", PILOT_MODE)

## 2. Discover currently listed free models

This cell queries OpenRouter's public model catalog using your key. It does not generate text. The `:free` suffix indicates a free offering in the catalog, but does not guarantee uptime or unlimited requests.

In [ ]:
catalog_response = requests.get(
    f"{BASE_URL}/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
print("Catalog HTTP:", catalog_response.status_code)
catalog_response.raise_for_status()

catalog_models = catalog_response.json().get("data", [])
free_models = sorted(
    [m for m in catalog_models if str(m.get("id", "")).endswith(":free")],
    key=lambda m: str(m.get("id", "")).lower()
)
free_model_ids = [m["id"] for m in free_models]

print("Free models currently listed:", len(free_model_ids))
display(pd.DataFrame([
    {"model_id": m.get("id"), "name": m.get("name")}
    for m in free_models
]))

## 3. Select the model lineup

The default lineup uses three IDs that were listed in the user's current catalog during setup. The notebook verifies each ID against the live catalog before testing. If a model is no longer listed, replace it with an ID shown in the previous cell.

In [ ]:
PREFERRED_MODEL_IDS = [
    "liquid/lfm-2.5-2.6b:free",
    "nvidia/nemotron-3.5-lightning:free",
    "apodex/apodex-1.1-mini:free",
]

MODEL_IDS = [model_id for model_id in PREFERRED_MODEL_IDS if model_id in free_model_ids]

if len(MODEL_IDS) < 3:
    for model_id in free_model_ids:
        if model_id not in MODEL_IDS:
            MODEL_IDS.append(model_id)
        if len(MODEL_IDS) >= 3:
            break

if not MODEL_IDS:
    raise RuntimeError("No free models were found in the current catalog.")

print("Models selected:")
for model_id in MODEL_IDS:
    print("-", model_id)

## 4. Create the benchmark set

The benchmark contains 30 scenarios across 10 settings and 3 prompt variations. Pilot mode uses the first 3 scenarios per model; full mode uses all 30 per model.

In [ ]:
SCENARIOS = [
    ("urban park", "A beginner exploring a public urban park for 30 minutes"),
    ("riverside path", "A beginner walking along a clearly marked public riverside path"),
    ("botanical garden", "A visitor exploring a public botanical garden for 45 minutes"),
    ("historic town", "A visitor discovering public historic streets without entering restricted areas"),
    ("neighborhood", "A person exploring a familiar neighborhood in daylight"),
    ("coastal promenade", "A beginner walking on a public coastal promenade, away from the water's edge"),
    ("forest trail", "A beginner on a marked, easy public trail in good weather"),
    ("city square", "A visitor exploring a public city square and nearby pedestrian streets"),
    ("lake path", "A beginner on a maintained public path around a lake"),
    ("community garden", "A visitor exploring a public community garden while respecting posted rules"),
]

VARIATIONS = [
    "Make the missions sensory and observation-based.",
    "Make the missions focus on noticing plants, architecture, and small details.",
    "Make the missions playful but accessible to a person with limited time.",
]

BENCHMARK_CASES = []
case_number = 1
for environment, scenario in SCENARIOS:
    for variation in VARIATIONS:
        BENCHMARK_CASES.append({
            "case_id": f"OA-{case_number:02d}",
            "environment": environment,
            "scenario": scenario,
            "variation": variation,
        })
        case_number += 1

assert len(BENCHMARK_CASES) == 30
display(pd.DataFrame(BENCHMARK_CASES).head(10))

## 5. Prompt, parse, and validate model responses

We request concise JSON and allow enough output tokens to reduce truncation. Some models may return reasoning without user-facing content or hit token limits; such responses are recorded as failures instead of being mistaken for successful JSON.

In [ ]:
SYSTEM_PROMPT = (
    "You generate practical outdoor adventures for a general audience. "
    "Return ONLY one valid JSON object, without Markdown fences or surrounding prose. "
    "Required schema: "
    '{"title": string, "description": string, "duration_minutes": integer, '
    '"missions": [{"title": string, "description": string, "xp": integer}], '
    '"total_xp": integer}. '
    "Create exactly 3 distinct missions. Use beginner-friendly activities in public, "
    "permitted spaces. Do not encourage trespassing, approaching wildlife, entering water, "
    "climbing structures, ignoring warnings, or leaving marked paths. Include a brief "
    "practical safety reminder in description. Keep the response concise."
)

UNSAFE_TERMS = [
    "trespass", "enter a restricted", "climb a building", "climb a bridge",
    "approach the wild animal", "feed the wild animal", "swim across",
    "walk into traffic", "ignore warning signs", "leave the marked trail",
]

def extract_json(text):
    if not isinstance(text, str) or not text.strip():
        return None
    candidate = text.strip()
    candidate = re.sub(r"^```(?:json)?\s*", "", candidate, flags=re.IGNORECASE)
    candidate = re.sub(r"\s*```$", "", candidate)
    try:
        obj = json.loads(candidate)
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        pass

    start, end = candidate.find("{"), candidate.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(candidate[start:end + 1])
            return obj if isinstance(obj, dict) else None
        except json.JSONDecodeError:
            return None
    return None

def get_text_content(message):
    # OpenRouter usually returns message.content, but some model/provider
    # combinations may return null or content in another shape.
    content = message.get("content")
    if isinstance(content, str) and content.strip():
        return content.strip()
    if isinstance(content, list):
        chunks = [
            part.get("text", "")
            for part in content
            if isinstance(part, dict) and isinstance(part.get("text"), str)
        ]
        joined = "\n".join(chunk for chunk in chunks if chunk.strip()).strip()
        if joined:
            return joined
    return None

def validate_output(raw_text):
    data = extract_json(raw_text)
    if data is None:
        return {
            "json_valid": False,
            "required_fields_present": False,
            "mission_count": None,
            "mission_count_valid": False,
            "xp_consistent": False,
            "unique_mission_title_ratio": None,
            "safety_flag_heuristic": None,
            "parsed_json": None,
        }

    required = {"title", "description", "duration_minutes", "missions", "total_xp"}
    missions = data.get("missions")
    fields_present = required.issubset(data.keys()) and isinstance(missions, list)
    mission_count = len(missions) if isinstance(missions, list) else 0

    titles = [
        str(m.get("title", "")).strip().lower()
        for m in missions
        if isinstance(m, dict) and str(m.get("title", "")).strip()
    ] if isinstance(missions, list) else []

    unique_ratio = len(set(titles)) / len(titles) if titles else 0.0

    xp_values = [
        m.get("xp") for m in missions
        if isinstance(m, dict) and isinstance(m.get("xp"), (int, float))
    ] if isinstance(missions, list) else []
    xp_consistent = (
        len(xp_values) == len(missions)
        and isinstance(data.get("total_xp"), (int, float))
        and sum(xp_values) == data.get("total_xp")
    ) if isinstance(missions, list) else False

    combined = json.dumps(data, ensure_ascii=False).lower()
    safety_flag = any(term in combined for term in UNSAFE_TERMS)

    return {
        "json_valid": True,
        "required_fields_present": fields_present,
        "mission_count": mission_count,
        "mission_count_valid": mission_count == 3,
        "xp_consistent": xp_consistent,
        "unique_mission_title_ratio": round(unique_ratio, 3),
        "safety_flag_heuristic": safety_flag,
        "parsed_json": data,
    }

## 6. Run a small pilot

Run 3 prompts per model first. We use a moderate token budget and record `finish_reason`, usage, and errors. A `200` response with no text is not counted as a successful output.

In [ ]:
def call_model(model_id, case):
    user_prompt = (
        f"Scenario: {case['scenario']}. "
        f"Environment category: {case['environment']}. "
        f"Style: {case['variation']} "
        "Return the required JSON schema exactly."
    )
    payload = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0,
        "max_tokens": 1400,
    }

    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        start = time.perf_counter()
        try:
            response = requests.post(
                f"{BASE_URL}/chat/completions",
                headers=HEADERS,
                json=payload,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            latency = round(time.perf_counter() - start, 3)

            if not response.ok:
                last_error = f"HTTP {response.status_code}: {response.text[:500]}"
                if response.status_code in (401, 402, 403, 429):
                    break
                if attempt < MAX_RETRIES:
                    time.sleep(2 ** attempt)
                    continue
                break

            body = response.json()
            choices = body.get("choices") or []
            if not choices:
                last_error = "Response had no choices"
                break

            choice = choices[0]
            message = choice.get("message") or {}
            content = get_text_content(message)
            finish_reason = choice.get("finish_reason")
            usage = body.get("usage") or {}

            if not content:
                # Record reasoning-only/empty responses as failures; do not
                # treat internal reasoning as the requested answer.
                last_error = f"No answer content; finish_reason={finish_reason}"
                return {
                    "model_id": model_id,
                    "case_id": case["case_id"],
                    "environment": case["environment"],
                    "status": "error",
                    "http_status": response.status_code,
                    "latency_seconds": latency,
                    "finish_reason": finish_reason,
                    "error": last_error,
                    "input_tokens": usage.get("prompt_tokens"),
                    "output_tokens": usage.get("completion_tokens"),
                    "raw_response": None,
                    **validate_output(None),
                }

            validation = validate_output(content)
            return {
                "model_id": model_id,
                "case_id": case["case_id"],
                "environment": case["environment"],
                "status": "ok" if validation["json_valid"] else "invalid_output",
                "http_status": response.status_code,
                "latency_seconds": latency,
                "finish_reason": finish_reason,
                "error": None if validation["json_valid"] else "Response was not valid JSON",
                "input_tokens": usage.get("prompt_tokens"),
                "output_tokens": usage.get("completion_tokens"),
                "raw_response": content,
                **validation,
            }

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {str(exc)[:300]}"
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)

    return {
        "model_id": model_id,
        "case_id": case["case_id"],
        "environment": case["environment"],
        "status": "error",
        "http_status": None,
        "latency_seconds": None,
        "finish_reason": None,
        "error": last_error,
        "input_tokens": None,
        "output_tokens": None,
        "raw_response": None,
        **validate_output(None),
    }

In [ ]:
run_cases = BENCHMARK_CASES[:PILOT_CASES if PILOT_MODE else FULL_CASES]
results = []

for model_id in MODEL_IDS:
    print(f"\nTesting model: {model_id}")
    for index, case in enumerate(run_cases, start=1):
        print(f"  {index}/{len(run_cases)} — {case['case_id']}", end=": ")
        result = call_model(model_id, case)
        results.append(result)
        print(result["status"], "|", result.get("error") or "")
        time.sleep(PAUSE_BETWEEN_REQUESTS_SECONDS)

results_df = pd.DataFrame(results)
display(results_df[[
    "model_id", "case_id", "status", "http_status", "latency_seconds",
    "finish_reason", "json_valid", "required_fields_present",
    "mission_count_valid", "xp_consistent", "error"
]])

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
results_path = f"outdoor_adventure_results_{timestamp}.csv"
results_df.to_csv(results_path, index=False)
print("Detailed results saved to:", results_path)

## 7. Summarize observed performance

These metrics describe only the requests actually run. Quality scores do not prove that the adventure is safe or enjoyable; manual review is required.

In [ ]:
def mean_or_none(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return round(float(values.mean()), 3) if len(values) else None

summary_rows = []
for model_id, group in results_df.groupby("model_id", dropna=False):
    summary_rows.append({
        "model_id": model_id,
        "requests_attempted": len(group),
        "successful_text_response_rate": round(group["raw_response"].notna().mean(), 3),
        "valid_json_rate_all_requests": round(group["json_valid"].fillna(False).mean(), 3),
        "required_fields_rate_all_requests": round(group["required_fields_present"].fillna(False).mean(), 3),
        "exactly_3_missions_rate": round(group["mission_count_valid"].fillna(False).mean(), 3),
        "xp_consistency_rate": round(group["xp_consistent"].fillna(False).mean(), 3),
        "mean_unique_mission_title_ratio": mean_or_none(group["unique_mission_title_ratio"]),
        "safety_heuristic_flag_rate": mean_or_none(group["safety_flag_heuristic"].astype(float)),
        "median_latency_seconds": mean_or_none(group["latency_seconds"]),
        "error_count": int(group["status"].eq("error").sum()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["valid_json_rate_all_requests", "exactly_3_missions_rate", "median_latency_seconds"],
    ascending=[False, False, True],
    na_position="last",
)
display(summary_df)

summary_path = f"outdoor_adventure_summary_{timestamp}.csv"
summary_df.to_csv(summary_path, index=False)
print("Summary saved to:", summary_path)

## 8. Inspect sample responses manually

Read examples and score them manually before making claims about quality. Suggested rubric (1–5): relevance, practicality, mission variety, accessibility, and safety. Keep the reviewer and scoring procedure documented.

In [ ]:
for model_id in MODEL_IDS:
    subset = results_df[
        (results_df["model_id"] == model_id) &
        results_df["raw_response"].notna()
    ]
    print("\n" + "=" * 90)
    print("MODEL:", model_id)
    if subset.empty:
        print("No text response succeeded in this run.")
    else:
        row = subset.iloc[0]
        print("CASE:", row["case_id"])
        print(row["raw_response"])

## 9. Run the full benchmark

If the pilot works and free-tier limits permit it:
1. Change `PILOT_MODE = True` to `PILOT_MODE = False` in Section 1.
2. Restart the kernel and run all cells in order.
3. Review any failures and the CSV outputs.
4. Record the actual run date, model IDs, number of requests, errors, and limitations.

The full run attempts up to 30 prompts per model (up to 90 calls for three models). Free availability and rate limits can change, so stop if you hit repeated rate limits.

## 10. Publish on Kaggle and write the DEV article

Upload the notebook and selected CSV results to Kaggle. Ensure the notebook does not contain the API key. Make the benchmark accessible to readers, then include its Kaggle URL in the DEV submission.

In the article, cover:
- What you measured and why
- Which exact model IDs you tested
- The real observed results and surprising findings
- What you would measure next
- The public Kaggle notebook link

Never claim that a model is "safest" based only on the keyword heuristic. Describe it as a basic screening check and explain its limitations.